In [1]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV

In [2]:
df = pd.read_csv("../../data/emi_prediction_dataset_cleaned.csv")
df.shape

(398455, 31)

In [3]:
X = df.drop(['emi_eligibility', 'max_monthly_emi'], axis=1)
y = df['emi_eligibility']

In [4]:
le = LabelEncoder()
y_encoded = le.fit_transform(df["emi_eligibility"])

In [5]:
X = df.drop(["emi_eligibility", "max_monthly_emi"], axis=1)
y = y_encoded

In [6]:
cat_cols = X.select_dtypes(include=['object']).columns
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [14]:
# xgb = XGBClassifier(
#     objective="multi:softprob",
#     random_state=42,
#     n_jobs=-1
# )

# param_dist = {
#     "n_estimators": [200, 300, 400],
#     "learning_rate": [0.05, 0.1],
#     "max_depth": [3, 4, 5, 6],
#     "min_child_weight": [1, 3, 5],
#     "subsample": [0.8, 1.0],
#     "colsample_bytree": [0.8, 1.0],
# }

# random_search = RandomizedSearchCV(
#     estimator=xgb,
#     param_distributions=param_dist,
#     n_iter=40,
#     scoring="f1_macro",
#     cv=3,
#     random_state=42,
#     n_jobs=-1
# ) 

# random_search.fit(X_train, y_train)
# print("Best Parameters:", random_search.best_params_)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42
)
xgb_model.fit(X_train, y_train)

In [13]:
y_pred_train = xgb_model.predict(X_train)
y_prob_train = xgb_model.predict_proba(X_train)
roc_auc_xgb = roc_auc_score(y_train, y_prob_train, multi_class="ovr")
print("Accuracy:", accuracy_score(y_train, y_pred_train))
print("\nClassification Report:\n", classification_report(y_train, y_pred_train))
print("\nConfusion Matrix:\n", confusion_matrix(y_train, y_pred_train))
print("ROC AUC Score:", roc_auc_xgb)

Accuracy: 0.9786268210964851

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.99      0.97     59058
           1       0.98      0.55      0.71     13891
           2       0.98      1.00      0.99    245815

    accuracy                           0.98    318764
   macro avg       0.97      0.85      0.89    318764
weighted avg       0.98      0.98      0.98    318764


Confusion Matrix:
 [[ 58730    115    213]
 [  2582   7670   3639]
 [   197     67 245551]]
ROC AUC Score: 0.998211317405915


In [ ]:
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)
roc_auc_xgb = roc_auc_score(y_test, y_prob, multi_class="ovr")


print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC AUC Score:", roc_auc_xgb)

Accuracy: 0.9610495539019462

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.98      0.95     14764
           1       0.80      0.23      0.36      3473
           2       0.97      1.00      0.99     61454

    accuracy                           0.96     79691
   macro avg       0.90      0.74      0.77     79691
weighted avg       0.96      0.96      0.95     79691


Confusion Matrix:
 [[14481   142   141]
 [ 1159   815  1499]
 [  104    59 61291]]
ROC AUC Score: 0.990454567294333
